In [7]:
from langchain.llms import OpenAI
from langchain.agents import create_pandas_dataframe_agent
from langchain.agents.agent_types import AgentType
from dotenv import load_dotenv
from sqlalchemy.engine import URL
import os
import psycopg2

load_dotenv()

True

In [5]:
client_id = 'cdb58009-6e46-493f-8610-41a37c30a029'

In [9]:
conn = psycopg2.connect(
    host=os.environ.get("HOST"),
    database="enqurious_etl_db",
    user=os.environ.get("USER"),
    password=os.environ.get("PASSWORD"),
)

In [10]:
cursor = conn.cursor()

In [18]:
try:
    cursor.execute(f"SELECT * FROM progress_fact WHERE client_id='{client_id}' and is_current=True;")
except Exception as e:
    conn.close()

In [19]:
result = cursor.fetchall()

In [20]:
columns = [desc[0] for desc in cursor.description]

In [14]:
import pandas as pd

In [21]:
progress_fact_df = pd.DataFrame(result, columns=columns)

In [24]:
progress_fact_df.columns

Index(['id', 'order_id', 'activity_id', 'activity_name', 'activity_status',
       'total_inputs_by_activity', 'completed_inputs_by_activity',
       'progress_in_percent_by_activity', 'duration_min_by_activity',
       'learner_status_by_order', 'project_id', 'project_name',
       'participants_email', 'participants_name', 'client_id',
       'activity_level', 'created_at', 'is_current'],
      dtype='object')

In [25]:
# now fetch the skills_fact data
try:
    cursor.execute(f"SELECT * FROM skills_fact WHERE client_id='{client_id}' and is_current=True;")
except Exception as e:
    conn.close()

In [26]:
result = cursor.fetchall()

In [27]:
columns = [desc[0] for desc in cursor.description]

In [28]:
skills_fact_df = pd.DataFrame(result, columns=columns)

In [32]:
skills_fact_df.columns

Index(['id', 'order_id', 'activity_name', 'activity_id', 'skill_name', 'score',
       'total', 'skill_tree_name', 'actual_hint_by_activity', 'hints_consumed',
       'client_id', 'project_id', 'project_name', 'participant_name',
       'participant_email', 'created_at', 'is_current'],
      dtype='object')

In [29]:
from langchain.chat_models import ChatOpenAI

In [38]:
# Set up the base template
prefix = """Answer the following questions as best you can, but speaking as a pirate might speak. 
If you don't get answer in df1 then search the answer in df2

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [df1, df2]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question"""


suffix = """Begin! Remember to speak as a pirate when giving your final answer. Use lots of "Arg"s

Question: {input}
{agent_scratchpad}"""

In [39]:
llm = ChatOpenAI(temperature=0, model='gpt-3.5-turbo-16k')

Keep one thing in mind, that don't provide the whole progress_fact and skills_fact_df. First convert that dataframe
into project level and then provide it

In [40]:
# Now train the agent based on this csv
agent = create_pandas_dataframe_agent(
    llm=llm,
    df=[progress_fact_df, skills_fact_df],
    verbose=True,
    agent_type=AgentType.OPENAI_FUNCTIONS,
    prefix=prefix
)

In [36]:
agent.run("Give the overall top 5 well performed learners of the week based on their score")



> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': "df1.groupby('participants_email')['score'].sum().nlargest(5)"}`


KeyError: 'Column not found: score'It seems that the column 'score' is not present in the dataframe df1. Could you please confirm the column name or provide the correct dataframe that contains the 'score' column?

> Finished chain.


"It seems that the column 'score' is not present in the dataframe df1. Could you please confirm the column name or provide the correct dataframe that contains the 'score' column?"

In [35]:
agent.run("How many columns are there in df1?")



> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': 'len(df1.columns)'}`


18There are 18 columns in df1.

> Finished chain.


'There are 18 columns in df1.'

In [37]:
agent.run("Refer the df2 and give the overall top 5 well performed learners of the week based on their score")



> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': "df2_sorted = df2.sort_values('score', ascending=False)\ntop_5_learners = df2_sorted.head(5)\ntop_5_learners"}`


                                                      id  \
39077  175ae55e-b8b3-466a-9331-806bd77edcc2_anirudh.v...   
39076  175ae55e-b8b3-466a-9331-806bd77edcc2_rohit.yad...   
39078  175ae55e-b8b3-466a-9331-806bd77edcc2_gedela.in...   
39074  175ae55e-b8b3-466a-9331-806bd77edcc2_sapna.bha...   
36903  175ae55e-b8b3-466a-9331-806bd77edcc2_nikhil.sh...   

                                   order_id  \
39077  175ae55e-b8b3-466a-9331-806bd77edcc2   
39076  175ae55e-b8b3-466a-9331-806bd77edcc2   
39078  175ae55e-b8b3-466a-9331-806bd77edcc2   
39074  175ae55e-b8b3-466a-9331-806bd77edcc2   
36903  175ae55e-b8b3-466a-9331-806bd77edcc2   

                                      activity_name  \
39077  Snowflake Data Cloud Features & Architecture   
39076  Snowflake Data Cloud Features & Architec

Retrying langchain.chat_models.openai.ChatOpenAI.completion_with_retry.<locals>._completion_with_retry in 4.0 seconds as it raised RateLimitError: Rate limit reached for default-gpt-3.5-turbo in organization org-bopjHrMEteRaVQsQwATYgdCP on requests per min. Limit: 3 / min. Please try again in 20s. Contact us through our help center at help.openai.com if you continue to have issues. Please add a payment method to your account to increase your rate limit. Visit https://platform.openai.com/account/billing to add a payment method..


InvalidRequestError: This model's maximum context length is 4097 tokens. However, your messages resulted in 4205 tokens (4125 in the messages, 80 in the functions). Please reduce the length of the messages or functions.

In [42]:
progress_fact_df.columns

Index(['id', 'order_id', 'activity_id', 'activity_name', 'activity_status',
       'total_inputs_by_activity', 'completed_inputs_by_activity',
       'progress_in_percent_by_activity', 'duration_min_by_activity',
       'learner_status_by_order', 'project_id', 'project_name',
       'participants_email', 'participants_name', 'client_id',
       'activity_level', 'created_at', 'is_current'],
      dtype='object')

In [48]:
# Now check the progress_fact df
progress_fact_project_level_df = progress_fact_df.groupby(
    [
        "project_id",
        "project_name",
        "order_id",
        "learner_status_by_order",
        "participants_email",
        "participants_name",
        "client_id",
        "activity_level",
        "created_at",
    ]
).agg(
    {
        "activity_id": "count",
        "total_inputs_by_activity": "sum",
        "completed_inputs_by_activity": "sum",
        "duration_min_by_activity": "sum"
    }
).reset_index()

In [53]:
progress_fact_project_level_df

,project_id,project_name,order_id,learner_status_by_order,participants_email,participants_name,client_id,activity_level,created_at,activity_id,total_inputs_by_activity,completed_inputs_by_activity,duration_min_by_activity
0,01b1b71f-ee68-4d4d-a7ac-31d7a1789120,SQL Assessment - Level 1,3a19451f-6b65-4a2f-92ca-a452b4b4df6c,Not started,rajendra.m@tredence.com,Rajendra M,cdb58009-6e46-493f-8610-41a37c30a029,Beginner,2023-07-03 08:35:32.091022,1,11,0.0,0.000000
1,01b1b71f-ee68-4d4d-a7ac-31d7a1789120,SQL Assessment - Level 1,6fcaf624-9aaa-4f1c-b856-6f3bfb739804,Not started,divya.nair@tredence.com,Divya S Nair,cdb58009-6e46-493f-8610-41a37c30a029,Beginner,2023-07-03 08:35:32.091022,1,11,0.0,0.000000
2,01b1b71f-ee68-4d4d-a7ac-31d7a1789120,SQL Assessment - Level 1,6fcaf624-9aaa-4f1c-b856-6f3bfb739804,Not started,rajendra.m@tredence.com,Rajendra M,cdb58009-6e46-493f-8610-41a37c30a029,Beginner,2023-07-03 08:35:32.091022,1,11,0.0,0.000000
3,01b1b71f-ee68-4d4d-a7ac-31d7a1789120,SQL Assessment - Level 1,ae5be33c-8573-4e4a-9aa9-ad60d8018733,Submitted,aditya.nilajkar@tredence.com,Aditya Nilajkar,cdb58009-6e46-493f-8610-41a37c30a029,Beginner,2023-07-03 08:35:32.091022,1,11,11.0,165.075130
4,01b1b71f-ee68-4d4d-a7ac-31d7a1789120,SQL Assessment - Level 1,ae5be33c-8573-4e4a-9aa9-ad60d8018733,Submitted,aftab.abbas@tredence.com,Aftab Abbas,cdb58009-6e46-493f-8610-41a37c30a029,Beginner,2023-07-03 08:35:32.091022,1,11,11.0,217.522507


In [49]:
progress_fact_project_level_df[progress_fact_project_level_df.duplicated()]

,project_id,project_name,order_id,learner_status_by_order,participants_email,participants_name,client_id,activity_level,created_at,activity_id,total_inputs_by_activity,completed_inputs_by_activity,duration_min_by_activity


In [59]:
# progress_fact_project_level_df.iloc[len(progress_fact_project_level_df)-1:, :]

In [60]:
progress_fact_project_level_df.columns

Index(['project_id', 'project_name', 'order_id', 'learner_status_by_order',
       'participants_email', 'participants_name', 'client_id',
       'activity_level', 'created_at', 'activity_id',
       'total_inputs_by_activity', 'completed_inputs_by_activity',
       'duration_min_by_activity'],
      dtype='object')

In [63]:
# rename the activity_id
progress_fact_project_level_df.rename(
    {
        "activity_id": "Total_activities",
        "total_inputs_by_activity": "Total_inputs",
        "completed_inputs_by_activity": "Completed_inputs",
    },
    axis=1,
    inplace=True
)

In [69]:
progress_fact_project_level_df['Completed_inputs'] = progress_fact_project_level_df['Completed_inputs'].astype(int)

In [83]:
# Now calculate the Progress (%)
progress_fact_project_level_df['Progress (%)'] = (
    progress_fact_project_level_df["Completed_inputs"]
    / progress_fact_project_level_df["Total_inputs"].replace(0, 1)
).map("{:.2%}".format)

In [89]:
progress_fact_project_level_df.shape

(6354, 14)

> The above dataframe is ready to provide as a context to agent

In [51]:
skills_fact_df.columns

Index(['id', 'order_id', 'activity_name', 'activity_id', 'skill_name', 'score',
       'total', 'skill_tree_name', 'actual_hint_by_activity', 'hints_consumed',
       'client_id', 'project_id', 'project_name', 'participant_name',
       'participant_email', 'created_at', 'is_current'],
      dtype='object')

In [88]:
# Now check the progress_fact df, to get unique hints
# skills_fact_activity_level_df = 
skills_fact_df.groupby(
    [
        "project_id",
        "project_name",
        "activity_id",
        "activity_name",
        "order_id",
        "participant_email",
        "participant_name",
        "client_id",
        "created_at",
    ]
).agg(
    {
        "skill_name": "count",
        "score": "sum",
        "total": "sum"
    }
).reset_index().rename({"skill_name": "Total_skills"})

,project_id,project_name,activity_id,activity_name,order_id,participant_email,participant_name,client_id,created_at,skill_name,score,total
0,01b1b71f-ee68-4d4d-a7ac-31d7a1789120,SQL Assessment - Level 1,36f38edb-c5d9-454c-b89e-b49f75ac7ee3,Hospital Networks in Country,ae5be33c-8573-4e4a-9aa9-ad60d8018733,aditya.nilajkar@tredence.com,Aditya Nilajkar,cdb58009-6e46-493f-8610-41a37c30a029,2023-07-03 08:35:44.765756,13,466.0,510.0
1,01b1b71f-ee68-4d4d-a7ac-31d7a1789120,SQL Assessment - Level 1,36f38edb-c5d9-454c-b89e-b49f75ac7ee3,Hospital Networks in Country,ae5be33c-8573-4e4a-9aa9-ad60d8018733,aftab.abbas@tredence.com,Aftab Abbas,cdb58009-6e46-493f-8610-41a37c30a029,2023-07-03 08:35:44.765756,13,465.0,510.0
2,01b1b71f-ee68-4d4d-a7ac-31d7a1789120,SQL Assessment - Level 1,36f38edb-c5d9-454c-b89e-b49f75ac7ee3,Hospital Networks in Country,ae5be33c-8573-4e4a-9aa9-ad60d8018733,akash.rotke@tredence.com,Akash Avdhut Rotke,cdb58009-6e46-493f-8610-41a37c30a029,2023-07-03 08:35:44.765756,13,413.0,510.0
3,01b1b71f-ee68-4d4d-a7ac-31d7a1789120,SQL Assessment - Level 1,36f38edb-c5d9-454c-b89e-b49f75ac7ee3,Hospital Networks in Country,ae5be33c-8573-4e4a-9aa9-ad60d8018733,amulya.kattemane@tredence.com,Amulya Kattemane,cdb58009-6e46-493f-8610-41a37c30a029,2023-07-03 08:35:44.765756,13,458.0,510.0
4,01b1b71f-ee68-4d4d-a7ac-31d7a1789120,SQL Assessment - Level 1,36f38edb-c5d9-454c-b89e-b49f75ac7ee3,Hospital Networks in Country,ae5be33c-8573-4e4a-9aa9-ad60d8018733,anupriya.dongare@tredence.com,Anupriya Vijay Dongare,cdb58009-6e46-493f-8610-41a37c30a029,2023-07-03 08:35:44.765756,13,489.0,510.0
...,...,...,...,...,...,...,...,...,...,...,...,...
6388,fbfe5eac-3fca-4a5d-a840-4550bb452f50,Practice Set | Python | Level 2,a4e3a958-f9a8-4140-9d68-b84f760ff665,Cricket Player Analysis,e213287b-c47e-43b2-9cdc-97dc65729f1c,tarun.gautam@tredence.com,Tarun Gautam,cdb58009-6e46-493f-8610-41a37c30a029,2023-07-19 21:33:49.462237,9,0.0,90.0
6389,fbfe5eac-3fca-4a5d-a840-4550bb452f50,Practice Set | Python | Level 2,a4e3a958-f9a8-4140-9d68-b84f760ff665,Cricket Player Analysis,e213287b-c47e-43b2-9cdc-97dc65729f1c,tejaswi.bajaj@tredence.com,Tejaswi Bajaj,cdb58009-6e46-493f-8610-41a37c30a029,2023-07-18 21:33:48.555971,9,0.0,90.0
6390,fbfe5eac-3fca-4a5d-a840-4550bb452f50,Practice Set | Python | Level 2,a4e3a958-f9a8-4140-9d68-b84f760ff665,Cricket Player Analysis,e213287b-c47e-43b2-9cdc-97dc65729f1c,tushar.sharma@tredence.com,Tushar Sharma,cdb58009-6e46-493f-8610-41a37c30a029,2023-07-19 21:33:49.462237,9,0.0,90.0
6391,fbfe5eac-3fca-4a5d-a840-4550bb452f50,Practice Set | Python | Level 2,a4e3a958-f9a8-4140-9d68-b84f760ff665,Cricket Player Analysis,e213287b-c47e-43b2-9cdc-97dc65729f1c,vanshika.gupta@tredence.com,Vanshika Gupta,cdb58009-6e46-493f-8610-41a37c30a029,2023-07-19 21:33:49.462237,9,0.0,90.0
